In [ ]:
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Lambda, Dense
import matplotlib.pyplot as plt

# ==========================================
# a. Data Preparation
# ==========================================

# Input corpus
raw_data = """Deep learning (also known as deep structured learning) is part of a broader family of machine learning methods based on artificial neural networks with representation learning. Learning can be supervised, semi-supervised or unsupervised. Deep-learning architectures such as deep neural networks, deep belief networks, deep reinforcement learning, recurrent neural networks, convolutional neural networks and Transformers have been applied to fields including computer vision, speech recognition, natural language processing, machine translation, bioinformatics, drug design, medical image analysis, climate science, material inspection and board game programs, where they have produced results comparable to and in some cases surpassing human expert performance."""

# Step 1: Split raw text into sentences
sentences = raw_data.split('.')

# Step 2: Clean each sentence (remove punctuation, single characters, and convert to lower case)
clean_sentences = []
for sentence in sentences:
    if sentence.strip() == "":
        continue
    sentence = re.sub(r'[^A-Za-z0-9]+', ' ', sentence)
    sentence = re.sub(r'(?:^| )\w (?:$| )', ' ', sentence).strip().lower()
    clean_sentences.append(sentence)

# Step 3: Tokenize text and build word-to-index mappings
tokenizer = Tokenizer()
tokenizer.fit_on_texts(clean_sentences)
sequences = tokenizer.texts_to_sequences(clean_sentences)

vocab_size = len(tokenizer.word_index) + 1
word_to_index = tokenizer.word_index
index_to_word = {i: w for w, i in word_to_index.items()}

# ==========================================
# b. Generate Training Data
# ==========================================

context_size = 2  # Window: 2 words left, 2 words right (total 4 context words)
X, Y = [], []

for seq in sequences:
    for i in range(context_size, len(seq) - context_size):
        target = seq[i]
        context = [seq[i - 2], seq[i - 1], seq[i + 1], seq[i + 2]]
        X.append(context)
        Y.append(target)

X = np.array(X)
Y = np.array(Y)

# ==========================================
# c. Train Model
# ==========================================

emb_size = 10  # Dimension of the embedding vectors

# CBOW Neural Network architecture
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=emb_size, input_length=2 * context_size),
    Lambda(lambda x: tf.reduce_mean(x, axis=1)),  # Average the context word embeddings
    Dense(256, activation='relu'),
    Dense(512, activation='relu'),
    Dense(vocab_size, activation='softmax')        # Predict the probability of target word
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Train the model and save the history for graphing
history = model.fit(X, Y, epochs=80, batch_size=32, verbose=0)

# ==========================================
# d. Output & Visualization
# ==========================================

def predict_target(context_words):
    """Predicts target center word from given 4 context words."""
    context_indices = [word_to_index[w] for w in context_words]
    x_test = np.array([context_indices])
    predictions = model.predict(x_test, verbose=0)
    best_index = np.argmax(predictions[0])
    return index_to_word.get(best_index)

# Test cases
test_sentences = [
    ["known", "as", "learning", "is"],                 # Target: "structured"
    ["cases", "surpassing", "expert", "performance"]   # Target: "human"
]

for context in test_sentences:
    prediction = predict_target(context)
    print(f"Context: {context} -> Predicted Target: {prediction}")

# Plotting loss and accuracy curves
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], color='red')
plt.title('Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], color='blue')
plt.title('Training Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.grid(True)

plt.tight_layout()
plt.show()